In [ ]:
import pandas as pd

# Load all three datasets
df_contracts = pd.read_csv('contracts.csv')
df_usage = pd.read_csv('usage.csv')
df_billing = pd.read_csv('billing.csv')

# Quick check
print("Contracts shape:", df_contracts.shape)
print("Usage shape:", df_usage.shape)
print("Billing shape:", df_billing.shape)

print("\nContracts preview:")
df_contracts.head(3)

In [ ]:
# ── Join all three datasets on account_id ──────────────────────────────
df = df_contracts.merge(df_usage, on='account_id').merge(df_billing, on='account_id')

print("Merged shape:", df.shape)

# ── Calculate leakage dollar amount ───────────────────────────────────
HOURLY_RATE = 0.50

df['leakage_usd'] = df.apply(
    lambda row: round(
        (row['actual_compute_hours'] - row['contracted_compute_hours']) * HOURLY_RATE, 2
    ) if row['overuse_flag'] else 0,
    axis=1
)

df['billing_gap_usd'] = round(df['expected_bill_usd'] - df['actual_bill_usd'], 2)

# ── Master leakage flag (any one of the three flags triggered) ─────────
df['any_leakage_flag'] = (
    df['overuse_flag'] |
    df['underbilled_flag'] |
    df['discount_violation_flag']
).astype(int)

# ── Summary ────────────────────────────────────────────────────────────
total_leakage = df['leakage_usd'].sum() + df['billing_gap_usd'].sum()

print(f"\nTotal estimated revenue leakage: ${total_leakage:,.2f}")
print(f"Accounts with any leakage flag:  {df['any_leakage_flag'].sum()}")
print(f"Overuse cases:                   {df['overuse_flag'].sum()}")
print(f"Underbilling cases:              {df['underbilled_flag'].sum()}")
print(f"Discount violations:             {df['discount_violation_flag'].sum()}")

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

# ── Recalculate derived columns ────────────────────────────────────────
df['usage_to_contract_ratio'] = df['actual_compute_hours'] / df['contracted_compute_hours']
df['billing_gap_pct'] = (df['billing_gap_usd'] / df['expected_bill_usd'].replace(0, 1)) * 100

# ── Features ───────────────────────────────────────────────────────────
features = [
    'usage_to_contract_ratio',
    'billing_gap_pct',
    'applied_discount_pct',
    'discount_ceiling_pct_x',
    'actual_compute_hours',
    'expected_bill_usd'
]

X = df[features]
y = df['any_leakage_flag']

# ── Model 1: Isolation Forest ──────────────────────────────────────────
iso = IsolationForest(contamination=0.15, random_state=42)
df['anomaly_score'] = iso.fit_predict(X)
df['anomaly_score'] = df['anomaly_score'].map({1: 0, -1: 1})

print("── Isolation Forest ──────────────────────────────")
print(f"Anomalies detected: {df['anomaly_score'].sum()} accounts\n")

# ── Model 2: Logistic Regression ───────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print("── Logistic Regression ───────────────────────────")
print(classification_report(y_test, y_pred))

# ── Risk score per account ─────────────────────────────────────────────
df['leakage_risk_pct'] = (lr.predict_proba(X)[:, 1] * 100).round(2)
df['risk_tier'] = pd.cut(
    df['leakage_risk_pct'],
    bins=[0, 40, 70, 100],
    labels=['Low', 'Medium', 'High']
)

print("── Risk Tier Breakdown ───────────────────────────")
print(df['risk_tier'].value_counts())

In [ ]:
# Export final dataset for Power BI
df.to_csv('oracle_leakage_final.csv', index=False)
print("✓ oracle_leakage_final.csv exported successfully")
print(f"  Shape: {df.shape}")
print(f"  Columns: {len(df.columns)}")